In [4]:
import os # For CUDA_LAUNCH_BLOCKING
# Ensure CUDA_LAUNCH_BLOCKING is set as the very first operation
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F # For softmax
import matplotlib.pyplot as plt

# --- Configuration for Self-Training ---
initial_labeled_ratio = 0.1 # Use 10% of (non-test) data as initially labeled
num_self_training_iterations = 5 # Max number of self-training rounds
confidence_threshold = 0.95 # Confidence to accept a pseudo-label
final_test_set_ratio = 0.2 # Hold out 20% of total data for final testing

# --- 0. Load Data ---
file_name = "Call Record Streams with Quality Evaluation and Cost Estimation (1).csv"
df = None
try:
    df = pd.read_csv(file_name)
    print(f"Successfully loaded '{file_name}' locally. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Local file '{file_name}' was not found. Attempting to load from Google Drive.")
    try:
         # Adjust if your path is different
        df = pd.read_csv(file_name)
        print(f"Successfully loaded '{file_name}' from Google Drive. Shape: {df.shape}")
    except Exception as e:
        print(f"Could not load data from Google Drive: {e}")
        df = pd.DataFrame()
except Exception as e:
    print(f"An error occurred during initial data loading: {e}")
    df = pd.DataFrame()

if df.empty:
   print("Data not loaded correctly. Process is stopping")
else: # Actual data processing
    print("\n--- 1. Define Target Variable & Initial Preprocessing ---")
    target_column = 'OverallCallQuality'
    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found in DataFrame.")

    df_processed = df.dropna(subset=[target_column]).copy()
    print(f"Shape after dropping NaNs in target '{target_column}': {df_processed.shape}")

    print(f"Unique values in '{target_column}' before any mapping: {df_processed[target_column].unique()}")

    # Use LabelEncoder directly on the original string target column for robust 0-indexed labels
    label_encoder = LabelEncoder()
    y_encoded_for_model = label_encoder.fit_transform(df_processed[target_column].astype(str))

    y_all_processed = pd.Series(y_encoded_for_model, index=df_processed.index)
    num_classes = len(label_encoder.classes_)

    print(f"Target variable '{target_column}' processed with LabelEncoder.")
    print(f"Original string labels mapped by LabelEncoder: {list(label_encoder.classes_)}")
    print(f"Final 0-indexed labels for model (unique): {np.unique(y_all_processed.values)}")
    print(f"Number of unique classes for model (num_classes): {num_classes}")
    print(f"Min label in final y_all_processed: {y_all_processed.min()}, Max label: {y_all_processed.max()}")

    # --- 2. Feature Selection & Cleaning (Applied to df_processed) ---
    columns_to_drop = [
        'StreamId', 'CallRecordId', 'Comment', 'SbcSessionId', 'CallerPhoneNumber',
        'CalleePhoneNumber', 'CallerIpAddress', 'CalleeIpAddress', 'CallerReflexiveIpAddress',
        'CalleeReflexiveIpAddress', 'CallerSubnet', 'CalleeSubnet',
        'SegmentFailedReason', 'SegmentFailureStage', 'CallerRelayIpAddress', 'CallerRelayPort',
        'CalleeRelayIpAddress', 'CalleeRelayPort', 'EstimatedGttCost', 'EstimatedSoftnetCost',
        'SbcEstimatedGttCost', 'SbcEstimatedSoftnetCost', 'CallStartTime', 'CallEndTime',
        'SbcSessionStartTime', 'SbcSessionEndTime', target_column,
        'SbcSessionStatus', 'Trunk', 'CallerPhoneNumberPrefix', 'CalleePhoneNumberPrefix', 'StreamQuality'
    ]

    df_features_all = df_processed.drop(columns=[col for col in columns_to_drop if col in df_processed.columns])

    if 'IsAudioForwardErrorCorrectionUsed' in df_features_all.columns:
        df_features_all['IsAudioForwardErrorCorrectionUsed'] = df_features_all['IsAudioForwardErrorCorrectionUsed'].astype(str).str.lower()
        df_features_all['IsAudioForwardErrorCorrectionUsed'] = df_features_all['IsAudioForwardErrorCorrectionUsed'].apply(lambda x: 1 if x == 'true' else 0).astype(int)

    # --- 3. Identify Feature Types (from df_features_all) ---
    numerical_features, categorical_features = [], []
    for col in df_features_all.columns:
        if df_features_all[col].dtype in ['int64', 'float64', 'int32', 'float32']:
            if df_features_all[col].nunique() < 20 and col not in ['PacketUtilization', 'DurationInSeconds', 'AverageJitter', 'MaxJitter']:
                 if col in ['CallFinalSipCode', 'CallEndSubReason']:
                     categorical_features.append(col); df_features_all[col] = df_features_all[col].astype(str)
                 else: numerical_features.append(col)
            else: numerical_features.append(col)
        elif df_features_all[col].dtype == 'bool':
            df_features_all[col] = df_features_all[col].astype(int); numerical_features.append(col)
        else:
            if df_features_all[col].nunique() < 50: categorical_features.append(col)
            else:
                print(f"Dropping high cardinality categorical column: '{col}' ({df_features_all[col].nunique()} unique values)")
                df_features_all.drop(columns=[col], inplace=True)

    if 'IsAudioForwardErrorCorrectionUsed' in df_features_all.columns and df_features_all['IsAudioForwardErrorCorrectionUsed'].dtype == 'int':
        if 'IsAudioForwardErrorCorrectionUsed' in categorical_features: categorical_features.remove('IsAudioForwardErrorCorrectionUsed')
        if 'IsAudioForwardErrorCorrectionUsed' not in numerical_features: numerical_features.append('IsAudioForwardErrorCorrectionUsed')

    numerical_features = [f for f in numerical_features if f in df_features_all.columns]
    categorical_features = [f for f in categorical_features if f in df_features_all.columns and f not in numerical_features]
    print(f"\nSelected Numerical Features ({len(numerical_features)}): {numerical_features if len(numerical_features) < 10 else str(numerical_features[:10]) + '...'}")
    print(f"Selected Categorical Features ({len(categorical_features)}): {categorical_features if len(categorical_features) < 10 else str(categorical_features[:10]) + '...'}")

    # --- 4. Create Preprocessing Pipelines & 5. Fit Preprocessor on ALL feature data ---
    numerical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

    cols_for_preprocessor = [col for col in df_features_all.columns if col in numerical_features or col in categorical_features]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_pipeline, [f for f in numerical_features if f in cols_for_preprocessor]),
            ('cat', categorical_pipeline, [f for f in categorical_features if f in cols_for_preprocessor])
        ],
        remainder='drop'
    )

    X_all_processed_np = preprocessor.fit_transform(df_features_all[cols_for_preprocessor])
    num_input_features = X_all_processed_np.shape[1]

    if np.isnan(X_all_processed_np).any(): print("WARNING: NaNs found in X_all_processed_np after preprocessing!")
    else: print("No NaNs found in X_all_processed_np after preprocessing.")
    print(f"\nShape of ALL processed features X: {X_all_processed_np.shape}")
    print(f"Shape of ALL target y: {y_all_processed.shape}")
    print(f"Number of input features for the CNN: {num_input_features}")

    # --- 6. Split Data for Self-Training Pipeline ---
    X_temp_ssl, X_test_final_np, y_temp_ssl, y_test_final_np = train_test_split(
        X_all_processed_np, y_all_processed.values,
        test_size=final_test_set_ratio,
        random_state=42,
        stratify=y_all_processed.values
    )
    print(f"Final Test Set (T) shape: X={X_test_final_np.shape}, y={y_test_final_np.shape}")

    try:
        X_unlabeled_current_np, X_labeled_initial_np, y_unlabeled_current_true_labels, y_labeled_initial_np = train_test_split(
            X_temp_ssl, y_temp_ssl,
            test_size=initial_labeled_ratio,
            random_state=42,
            stratify=y_temp_ssl
        )
    except ValueError as e:
        print(f"Stratification failed for initial L/U split: {e}. Splitting without stratification.")
        X_unlabeled_current_np, X_labeled_initial_np, y_unlabeled_current_true_labels, y_labeled_initial_np = train_test_split(
            X_temp_ssl, y_temp_ssl,
            test_size=initial_labeled_ratio,
            random_state=42
        )

    print(f"Initial Labeled Set (L) shape: X={X_labeled_initial_np.shape}, y={y_labeled_initial_np.shape}")
    print(f"Initial Unlabeled Set (U) shape: X={X_unlabeled_current_np.shape} (true labels hidden for training)")

    assert y_labeled_initial_np.min() >= 0 and y_labeled_initial_np.max() < num_classes
    assert y_unlabeled_current_true_labels.min() >= 0 and y_unlabeled_current_true_labels.max() < num_classes
    assert y_test_final_np.min() >= 0 and y_test_final_np.max() < num_classes
    print("Label range checks passed for L, U (true), and T splits.")

    X_labeled_current_np = X_labeled_initial_np.copy()
    y_labeled_current_np = y_labeled_initial_np.copy()


# --- PyTorch Specific Part (Only if data was processed) ---
if 'X_labeled_current_np' in globals():
    batch_size = 64 # Define batch_size here for all DataLoaders in this block

    X_test_final_tensor = torch.tensor(X_test_final_np, dtype=torch.float32).unsqueeze(1)
    y_test_final_tensor = torch.tensor(y_test_final_np, dtype=torch.long)
    test_final_dataset = TensorDataset(X_test_final_tensor, y_test_final_tensor)
    test_final_loader = DataLoader(test_final_dataset, batch_size=batch_size, shuffle=False) # Use variable

    class CNN1D(nn.Module):
        def __init__(self, input_features, num_classes_model):
            super(CNN1D, self).__init__()
            self.conv1 = nn.Conv1d(1,32,kernel_size=3,padding=1); self.bn1=nn.BatchNorm1d(32); self.relu1=nn.ReLU(); self.pool1=nn.MaxPool1d(2,2); self.dropout1=nn.Dropout(0.25)
            self.conv2 = nn.Conv1d(32,64,kernel_size=3,padding=1); self.bn2=nn.BatchNorm1d(64); self.relu2=nn.ReLU(); self.pool2=nn.MaxPool1d(2,2); self.dropout2=nn.Dropout(0.25)
            bn1_original_mode=self.bn1.training; bn2_original_mode=self.bn2.training
            self.bn1.eval(); self.bn2.eval()
            with torch.no_grad():
                dummy_input = torch.randn(1,1,input_features)
                x = self.conv1(dummy_input); x=self.bn1(x); x=self.relu1(x); x=self.pool1(x)
                x = self.conv2(x); x=self.bn2(x); x=self.relu2(x); x=self.pool2(x)
                flattened_size = x.shape[1]*x.shape[2]
                if flattened_size <= 0:
                    self.bn1.train(bn1_original_mode); self.bn2.train(bn2_original_mode)
                    raise ValueError(f"Flattened size {flattened_size} invalid.")
            self.bn1.train(bn1_original_mode); self.bn2.train(bn2_original_mode)
            self.flatten=nn.Flatten(); self.fc1=nn.Linear(flattened_size,128)
            self.bn3=nn.BatchNorm1d(128); self.relu3=nn.ReLU(); self.dropout3=nn.Dropout(0.5)
            self.fc2=nn.Linear(128,num_classes_model)
        def forward(self,x):
            x=self.conv1(x); x=self.bn1(x); x=self.relu1(x); x=self.pool1(x); x=self.dropout1(x)
            x=self.conv2(x); x=self.bn2(x); x=self.relu2(x); x=self.pool2(x); x=self.dropout2(x)
            x=self.flatten(x)
            x=self.fc1(x); x=self.bn3(x); x=self.relu3(x); x=self.dropout3(x)
            x=self.fc2(x)
            return x

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    print("\n--- Starting Self-Training Pipeline ---")
    for iteration in range(num_self_training_iterations):
        print(f"\n--- Self-Training Iteration: {iteration + 1}/{num_self_training_iterations} ---")
        print(f"Current Labeled Set size: {X_labeled_current_np.shape[0]}")
        print(f"Current Unlabeled Set size: {X_unlabeled_current_np.shape[0]}")

        if X_labeled_current_np.shape[0] == 0: print("No labeled data. Stopping."); break
        if X_unlabeled_current_np.shape[0] == 0: print("No unlabeled data. Stopping."); break

        if X_labeled_current_np.shape[0] > 1:
            val_size_iter = min(0.2, 500 / X_labeled_current_np.shape[0])
            val_size_iter = max(val_size_iter, 0.1) if X_labeled_current_np.shape[0] > 10 else 0
            if val_size_iter > 0 and X_labeled_current_np.shape[0] * val_size_iter >= num_classes :
                try:
                    X_train_iter_np, X_val_iter_np, y_train_iter_np, y_val_iter_np = train_test_split(
                        X_labeled_current_np, y_labeled_current_np, test_size=val_size_iter, random_state=42+iteration, stratify=y_labeled_current_np)
                except ValueError:
                     X_train_iter_np, X_val_iter_np, y_train_iter_np, y_val_iter_np = train_test_split(
                        X_labeled_current_np, y_labeled_current_np, test_size=val_size_iter, random_state=42+iteration)
            elif val_size_iter > 0 :
                 X_train_iter_np, X_val_iter_np, y_train_iter_np, y_val_iter_np = train_test_split(
                        X_labeled_current_np, y_labeled_current_np, test_size=val_size_iter, random_state=42+iteration)
            else:
                X_train_iter_np, y_train_iter_np = X_labeled_current_np, y_labeled_current_np
                X_val_iter_np, y_val_iter_np = np.array([]).reshape(0,num_input_features), np.array([])
        else:
            X_train_iter_np, y_train_iter_np = X_labeled_current_np, y_labeled_current_np
            X_val_iter_np, y_val_iter_np = np.array([]).reshape(0,num_input_features), np.array([])

        X_train_iter = torch.tensor(X_train_iter_np, dtype=torch.float32).unsqueeze(1)
        y_train_iter = torch.tensor(y_train_iter_np, dtype=torch.long)
        train_iter_dataset = TensorDataset(X_train_iter, y_train_iter)
        train_iter_loader = DataLoader(train_iter_dataset, batch_size=batch_size, shuffle=True) # Use variable

        val_iter_loader = None
        if X_val_iter_np.shape[0] > 0:
            X_val_iter = torch.tensor(X_val_iter_np, dtype=torch.float32).unsqueeze(1)
            y_val_iter = torch.tensor(y_val_iter_np, dtype=torch.long)
            val_iter_dataset = TensorDataset(X_val_iter, y_val_iter)
            val_iter_loader = DataLoader(val_iter_dataset, batch_size=batch_size, shuffle=False) # Use variable

        model_iter = CNN1D(input_features=num_input_features, num_classes_model=num_classes).to(device)
        optimizer_iter = optim.Adam(model_iter.parameters(), lr=0.001)
        criterion_iter = nn.CrossEntropyLoss()

        print(f"Training model for iteration {iteration + 1} on {X_train_iter_np.shape[0]} labeled samples...")
        epochs_per_iteration = 50
        for epoch in range(epochs_per_iteration):
            model_iter.train()
            for inputs, labels in train_iter_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer_iter.zero_grad(); outputs = model_iter(inputs); loss = criterion_iter(outputs, labels)
                if torch.isnan(loss): print(f"NaN loss iter {iteration+1}, epoch {epoch+1}. Skipping."); continue
                loss.backward(); optimizer_iter.step()
            if val_iter_loader and (epoch % 5 == 0 or epoch == epochs_per_iteration -1) :
                model_iter.eval(); val_loss_inner, val_acc_inner = 0,0
                with torch.no_grad():
                    for inputs, labels in val_iter_loader:
                        inputs, labels = inputs.to(device), labels.to(device)
                        outputs = model_iter(inputs); val_loss_inner += criterion_iter(outputs,labels).item()*inputs.size(0)
                        _, predicted = torch.max(outputs.data,1); val_acc_inner += (predicted==labels).sum().item()
                val_loss_inner /= len(val_iter_loader.dataset); val_acc_inner /= len(val_iter_loader.dataset)
                print(f"  Iter {iteration+1}, Epoch {epoch+1}: Val Loss: {val_loss_inner:.4f}, Val Acc: {val_acc_inner:.4f}")

        print(f"Predicting on {X_unlabeled_current_np.shape[0]} unlabeled samples...")
        X_unlabeled_tensor = torch.tensor(X_unlabeled_current_np, dtype=torch.float32).unsqueeze(1).to(device)
        unlabeled_dataset_iter = TensorDataset(X_unlabeled_tensor)
        unlabeled_loader_iter = DataLoader(unlabeled_dataset_iter, batch_size=batch_size, shuffle=False) # Use variable

        model_iter.eval(); all_pseudo_labels, all_confidences = [], []
        with torch.no_grad():
            for (inputs,) in unlabeled_loader_iter:
                outputs = model_iter(inputs); probabilities = F.softmax(outputs,dim=1)
                confidences, pseudo_labels = torch.max(probabilities,dim=1)
                all_pseudo_labels.extend(pseudo_labels.cpu().numpy()); all_confidences.extend(confidences.cpu().numpy())
        all_pseudo_labels = np.array(all_pseudo_labels); all_confidences = np.array(all_confidences)
        high_confidence_indices = np.where(all_confidences >= confidence_threshold)[0]
        num_new_pseudo_labels = len(high_confidence_indices)
        print(f"Found {num_new_pseudo_labels} new pseudo-labels with confidence >= {confidence_threshold}")

        if num_new_pseudo_labels == 0 and iteration > 0: print("No new pseudo-labels. Stopping."); break
        if num_new_pseudo_labels > 0:
            newly_labeled_features = X_unlabeled_current_np[high_confidence_indices]
            newly_pseudo_labels = all_pseudo_labels[high_confidence_indices]
            X_labeled_current_np = np.concatenate((X_labeled_current_np,newly_labeled_features),axis=0)
            y_labeled_current_np = np.concatenate((y_labeled_current_np,newly_pseudo_labels),axis=0)
            X_unlabeled_current_np = np.delete(X_unlabeled_current_np,high_confidence_indices,axis=0)
            y_unlabeled_current_true_labels = np.delete(y_unlabeled_current_true_labels,high_confidence_indices,axis=0)
        final_model_for_evaluation = model_iter

    print("\n--- Final Model Evaluation on Hold-Out Test Set ---")
    if 'final_model_for_evaluation' in locals() and X_test_final_np.shape[0] > 0:
        final_model_for_evaluation.eval(); test_correct,test_total,test_loss_total = 0,0,0
        all_final_preds, all_final_labels = [],[]
        with torch.no_grad():
            for inputs, labels in test_final_loader:
                inputs,labels = inputs.to(device),labels.to(device); outputs = final_model_for_evaluation(inputs)
                loss = criterion_iter(outputs,labels); test_loss_total += loss.item()*inputs.size(0)
                _, predicted = torch.max(outputs.data,1); test_total += labels.size(0)
                test_correct += (predicted==labels).sum().item()
                all_final_preds.extend(predicted.cpu().numpy()); all_final_labels.extend(labels.cpu().numpy())
        avg_test_loss=test_loss_total/len(test_final_loader.dataset) if len(test_final_loader.dataset)>0 else 0
        test_accuracy=test_correct/test_total if test_total > 0 else 0
        print(f"Final Test Loss: {avg_test_loss:.4f}, Final Test Accuracy: {test_accuracy:.4f}")
        from sklearn.metrics import classification_report
        print("\nClassification Report on Final Test Set:")
        if label_encoder is not None:
            report_target_names = list(label_encoder.classes_)
            report_indices = np.arange(num_classes)
            print(classification_report(all_final_labels, all_final_preds, labels=report_indices, target_names=report_target_names, zero_division=0))
        else:
            print("LabelEncoder not available. Printing report with numeric labels.")
            print(classification_report(all_final_labels, all_final_preds, zero_division=0))
    else: print("Final model not available or test set is empty. Skipping final evaluation.")
    print("\n--- PyTorch Self-Training CNN Implementation Complete ---")
else:
    print("PyTorch model training and evaluation skipped as data was not loaded/processed successfully or dummy data was used without full SSL setup.")



C:\Users\leduc\AppData\Local\Temp\ipykernel_10356\3418786930.py:29: DtypeWarning: Columns (31,32,38,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name)


Successfully loaded 'Call Record Streams with Quality Evaluation and Cost Estimation (1).csv' locally. Shape: (35028, 63)

--- 1. Define Target Variable & Initial Preprocessing ---
Shape after dropping NaNs in target 'OverallCallQuality': (28202, 63)
Unique values in 'OverallCallQuality' before any mapping: ['Poor' 'Acceptable' 'Good']
Target variable 'OverallCallQuality' processed with LabelEncoder.
Original string labels mapped by LabelEncoder: ['Acceptable', 'Good', 'Poor']
Final 0-indexed labels for model (unique): [0 1 2]
Number of unique classes for model (num_classes): 3
Min label in final y_all_processed: 0, Max label: 2

Selected Numerical Features (21): ['IsStreamDirectionToCallee', 'PacketUtilization', 'AverageAudioDegradation', 'AverageAudioNetworkJitter', 'AverageBandwidthEstimate', 'AverageJitter', 'AveragePacketLossRate', 'AverageRatioOfConcealedSamples', 'AverageRoundTripTime', 'IsAudioForwardErrorCorrectionUsed']...
Selected Categorical Features (10): ['AudioCodec', 'C

KeyboardInterrupt: 

In [3]:
import os # For CUDA_LAUNCH_BLOCKING
# Ensure CUDA_LAUNCH_BLOCKING is set as the very first operation
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F # For softmax
import matplotlib.pyplot as plt

# --- Configuration for Self-Training ---
initial_labeled_ratio = 0.1 # Use 10% of (non-test) data as initially labeled
num_self_training_iterations = 5 # Max number of self-training rounds
confidence_threshold = 0.95 # Confidence to accept a pseudo-label
final_test_set_ratio = 0.2 # Hold out 20% of total data for final testing

# --- 0. Load Data ---
file_name = "Call Record Streams with Quality Evaluation and Cost Estimation (1).csv"
df = None
try:
    df = pd.read_csv(file_name)
    print(f"Successfully loaded '{file_name}' locally. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Local file '{file_name}' was not found. Attempting to load from Google Drive.")
    try:
         # Adjust if your path is different
        df = pd.read_csv(file_name)
        print(f"Successfully loaded '{file_name}' from Google Drive. Shape: {df.shape}")
    except Exception as e:
        print(f"Could not load data from Google Drive: {e}")
        df = pd.DataFrame()
except Exception as e:
    print(f"An error occurred during initial data loading: {e}")
    df = pd.DataFrame()

if df.empty:
   print("Data not loaded correctly. Process is stopping")
else: # Actual data processing
    print("\n--- 1. Define Target Variable & Initial Preprocessing ---")
    target_column = 'OverallCallQuality'
    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found in DataFrame.")

    df_processed = df.dropna(subset=[target_column]).copy()
    print(f"Shape after dropping NaNs in target '{target_column}': {df_processed.shape}")

    print(f"Unique values in '{target_column}' before any mapping: {df_processed[target_column].unique()}")

    # Use LabelEncoder directly on the original string target column for robust 0-indexed labels
    label_encoder = LabelEncoder()
    y_encoded_for_model = label_encoder.fit_transform(df_processed[target_column].astype(str))

    y_all_processed = pd.Series(y_encoded_for_model, index=df_processed.index)
    num_classes = len(label_encoder.classes_)

    print(f"Target variable '{target_column}' processed with LabelEncoder.")
    print(f"Original string labels mapped by LabelEncoder: {list(label_encoder.classes_)}")
    print(f"Final 0-indexed labels for model (unique): {np.unique(y_all_processed.values)}")
    print(f"Number of unique classes for model (num_classes): {num_classes}")
    print(f"Min label in final y_all_processed: {y_all_processed.min()}, Max label: {y_all_processed.max()}")

    # --- 2. Feature Selection & Cleaning (Applied to df_processed) ---
    columns_to_drop = [
        'StreamId', 'CallRecordId', 'Comment', 'SbcSessionId', 'CallerPhoneNumber',
        'CalleePhoneNumber', 'CallerIpAddress', 'CalleeIpAddress', 'CallerReflexiveIpAddress',
        'CalleeReflexiveIpAddress', 'CallerSubnet', 'CalleeSubnet',
        'SegmentFailedReason', 'SegmentFailureStage', 'CallerRelayIpAddress', 'CallerRelayPort',
        'CalleeRelayIpAddress', 'CalleeRelayPort', 'EstimatedGttCost', 'EstimatedSoftnetCost',
        'SbcEstimatedGttCost', 'SbcEstimatedSoftnetCost', 'CallStartTime', 'CallEndTime',
        'SbcSessionStartTime', 'SbcSessionEndTime', target_column,
        'SbcSessionStatus', 'Trunk', 'CallerPhoneNumberPrefix', 'CalleePhoneNumberPrefix', 'StreamQuality'
    ]

    df_features_all = df_processed.drop(columns=[col for col in columns_to_drop if col in df_processed.columns])

    if 'IsAudioForwardErrorCorrectionUsed' in df_features_all.columns:
        df_features_all['IsAudioForwardErrorCorrectionUsed'] = df_features_all['IsAudioForwardErrorCorrectionUsed'].astype(str).str.lower()
        df_features_all['IsAudioForwardErrorCorrectionUsed'] = df_features_all['IsAudioForwardErrorCorrectionUsed'].apply(lambda x: 1 if x == 'true' else 0).astype(int)

    # --- 3. Identify Feature Types (from df_features_all) ---
    numerical_features, categorical_features = [], []
    for col in df_features_all.columns:
        if df_features_all[col].dtype in ['int64', 'float64', 'int32', 'float32']:
            if df_features_all[col].nunique() < 20 and col not in ['PacketUtilization', 'DurationInSeconds', 'AverageJitter', 'MaxJitter']:
                 if col in ['CallFinalSipCode', 'CallEndSubReason']:
                     categorical_features.append(col); df_features_all[col] = df_features_all[col].astype(str)
                 else: numerical_features.append(col)
            else: numerical_features.append(col)
        elif df_features_all[col].dtype == 'bool':
            df_features_all[col] = df_features_all[col].astype(int); numerical_features.append(col)
        else:
            if df_features_all[col].nunique() < 50: categorical_features.append(col)
            else:
                print(f"Dropping high cardinality categorical column: '{col}' ({df_features_all[col].nunique()} unique values)")
                df_features_all.drop(columns=[col], inplace=True)

    if 'IsAudioForwardErrorCorrectionUsed' in df_features_all.columns and df_features_all['IsAudioForwardErrorCorrectionUsed'].dtype == 'int':
        if 'IsAudioForwardErrorCorrectionUsed' in categorical_features: categorical_features.remove('IsAudioForwardErrorCorrectionUsed')
        if 'IsAudioForwardErrorCorrectionUsed' not in numerical_features: numerical_features.append('IsAudioForwardErrorCorrectionUsed')

    numerical_features = [f for f in numerical_features if f in df_features_all.columns]
    categorical_features = [f for f in categorical_features if f in df_features_all.columns and f not in numerical_features]
    print(f"\nSelected Numerical Features ({len(numerical_features)}): {numerical_features if len(numerical_features) < 10 else str(numerical_features[:10]) + '...'}")
    print(f"Selected Categorical Features ({len(categorical_features)}): {categorical_features if len(categorical_features) < 10 else str(categorical_features[:10]) + '...'}")

    # --- 4. Create Preprocessing Pipelines & 5. Fit Preprocessor on ALL feature data ---
    numerical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

    cols_for_preprocessor = [col for col in df_features_all.columns if col in numerical_features or col in categorical_features]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_pipeline, [f for f in numerical_features if f in cols_for_preprocessor]),
            ('cat', categorical_pipeline, [f for f in categorical_features if f in cols_for_preprocessor])
        ],
        remainder='drop'
    )

    X_all_processed_np = preprocessor.fit_transform(df_features_all[cols_for_preprocessor])
    num_input_features = X_all_processed_np.shape[1]
    print(f"Num input features after preprocessing: {num_input_features}")

C:\Users\leduc\AppData\Local\Temp\ipykernel_10356\1876999310.py:29: DtypeWarning: Columns (31,32,38,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name)


Successfully loaded 'Call Record Streams with Quality Evaluation and Cost Estimation (1).csv' locally. Shape: (35028, 63)

--- 1. Define Target Variable & Initial Preprocessing ---
Shape after dropping NaNs in target 'OverallCallQuality': (28202, 63)
Unique values in 'OverallCallQuality' before any mapping: ['Poor' 'Acceptable' 'Good']
Target variable 'OverallCallQuality' processed with LabelEncoder.
Original string labels mapped by LabelEncoder: ['Acceptable', 'Good', 'Poor']
Final 0-indexed labels for model (unique): [0 1 2]
Number of unique classes for model (num_classes): 3
Min label in final y_all_processed: 0, Max label: 2

Selected Numerical Features (21): ['IsStreamDirectionToCallee', 'PacketUtilization', 'AverageAudioDegradation', 'AverageAudioNetworkJitter', 'AverageBandwidthEstimate', 'AverageJitter', 'AveragePacketLossRate', 'AverageRatioOfConcealedSamples', 'AverageRoundTripTime', 'IsAudioForwardErrorCorrectionUsed']...
Selected Categorical Features (10): ['AudioCodec', 'C